In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hdbscan
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
from pathlib import Path

plt.style.use("seaborn-v0_8")

In [2]:
df = pd.read_csv("/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/processed/fotos_canarias.csv")

df = df.dropna(subset=["latitude", "longitude"]).copy()

df["datetaken"] = pd.to_datetime(df["datetaken"], errors="coerce")

df = df[df["accuracy"].fillna(0) >= 15].copy()

df = df.drop_duplicates(subset=["owner", "latitude", "longitude", "datetaken"])



In [3]:
centroides = (
    df.groupby("isla")[["latitude", "longitude"]]
      .median()
      .to_dict("index")
)

def asignar_isla_por_centroides(lat, lon, centroides):
    best, dmin = None, 1e9
    for isla, coords in centroides.items():
        d = (lat - coords["latitude"])**2 + (lon - coords["longitude"])**2
        if d < dmin:
            best, dmin = isla, d
    return best

if "isla_fix" not in df.columns:
    df["isla_fix"] = df.apply(
        lambda r: r["isla"] if pd.notna(r["isla"])
                  else asignar_isla_por_centroides(r.latitude, r.longitude, centroides),
        axis=1
    )

In [4]:
R_EARTH_M = 6371000.0

def r90_metros(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(np.clip(
        np.sin(lat0)*np.sin(pts[:,0]) +
        np.cos(lat0)*np.cos(pts[:,0])*np.cos(pts[:,1]-lon0),
        -1.0, 1.0
    ))
    return np.percentile(d * R_EARTH_M, 90)

In [5]:
df["cluster_hdbscan"] = -1
df["hdbscan_prob"] = np.nan
df["hdbscan_outlier_score"] = np.nan

BASE_MIN_CLUSTER = 25
BASE_MIN_SAMPLES = 15

for isla in sorted(df["isla_fix"].unique()):
    sub = df[df["isla_fix"] == isla]
    if sub.empty:
        continue

    coords_r = np.radians(sub[["latitude","longitude"]].values)

    clusterer = hdbscan.HDBSCAN(
        metric="haversine",
        min_cluster_size=BASE_MIN_CLUSTER,
        min_samples=BASE_MIN_SAMPLES,
        cluster_selection_method="leaf",
        allow_single_cluster=True
    )

    labels = clusterer.fit_predict(coords_r)

    df.loc[sub.index, "cluster_hdbscan"] = labels
    df.loc[sub.index, "hdbscan_prob"] = clusterer.probabilities_
    out = getattr(clusterer, "outlier_scores_", None)
    if out is not None:
        df.loc[sub.index, "hdbscan_outlier_score"] = out

C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar

In [6]:
from itertools import product

grid_min_cluster = [20, 25, 30]
grid_min_samples = [10, 15, 20]
grid_selection   = ["eom", "leaf"]

def eval_isla(sub, mcs, ms, sel):
    coords_r = np.radians(sub[["latitude","longitude"]].values)

    cl = hdbscan.HDBSCAN(
        metric="haversine",
        min_cluster_size=mcs,
        min_samples=ms,
        cluster_selection_method=sel,
        allow_single_cluster=True
    )

    labels = cl.fit_predict(coords_r)
    prob   = cl.probabilities_

    ruido = (labels == -1).mean()
    k = len(set(labels) - {-1})

    r90_vals = []
    for lab in set(labels) - {-1}:
        grp = sub[labels == lab]
        r90_vals.append(r90_metros(grp))

    return {
        "ruido": ruido,
        "k": k,
        "r90_med": np.median(r90_vals) if r90_vals else np.nan,
        "prob_mean": prob[labels != -1].mean() if np.any(labels != -1) else np.nan,
        "min_cluster_size": mcs,
        "min_samples": ms,
        "selection": sel
    }

best_by_isla = {}

for isla in sorted(df["isla_fix"].unique()):
    sub = df[df["isla_fix"] == isla][["latitude","longitude","id"]]
    if len(sub) < 50:
        continue

    resultados = [
        eval_isla(sub, mcs, ms, sel)
        for mcs, ms, sel in product(grid_min_cluster, grid_min_samples, grid_selection)
    ]

    cand = pd.DataFrame(resultados)
    cand = cand.sort_values(["ruido", "r90_med", "prob_mean"],
                            ascending=[True, True, False])

    best_by_isla[isla] = cand.iloc[0].to_dict()

KeyboardInterrupt: 

In [13]:
df["cluster_hdbscan_opt"] = -1
df["hdbscan_prob_opt"] = np.nan
df["hdbscan_outlier_opt"] = np.nan

for isla, params in best_by_isla.items():
    sub = df[df["isla_fix"] == isla]
    coords_r = np.radians(sub[["latitude","longitude"]].values)

    clusterer = hdbscan.HDBSCAN(
        metric="haversine",
        min_cluster_size=int(params["min_cluster_size"]),
        min_samples=int(params["min_samples"]),
        cluster_selection_method=params["selection"],
        allow_single_cluster=True
    )

    labels = clusterer.fit_predict(coords_r)

    df.loc[sub.index, "cluster_hdbscan_opt"] = labels
    df.loc[sub.index, "hdbscan_prob_opt"] = clusterer.probabilities_

    out = getattr(clusterer, "outlier_scores_", None)
    if out is not None:
        df.loc[sub.index, "hdbscan_outlier_opt"] = out



C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar divide
  self._outlier_scores = outlier_scores(self._condensed_tree)
C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\.venv\Lib\site-packages\hdbscan\hdbscan_.py:1509: RuntimeWarning: invalid value encountered in scalar

In [14]:
MIN_CLUSTER_SIZE_FINAL = 10

sizes = df[df["cluster_hdbscan_opt"] != -1].groupby("cluster_hdbscan_opt").size()
small = sizes[sizes < MIN_CLUSTER_SIZE_FINAL].index

df.loc[df["cluster_hdbscan_opt"].isin(small), "cluster_hdbscan_opt"] = -1

PERC_OUT = 90

for isla in df["isla_fix"].unique():
    sub = df[(df["isla_fix"] == isla) & (df["cluster_hdbscan_opt"] != -1)]
    if sub.empty:
        continue
    thr = np.nanpercentile(sub["hdbscan_outlier_opt"], PERC_OUT)
    idx = sub.index[sub["hdbscan_outlier_opt"] > thr]
    df.loc[idx, "cluster_hdbscan_opt"] = -1

In [15]:
df_export = df.copy()

if "cluster_hdbscan" in df_export.columns:
    df_export = df_export.drop(columns=["cluster_hdbscan"])

df_export = df_export.rename(columns={"cluster_hdbscan_opt": "cluster_hdbscan"})

df_export.to_csv(
    "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_hdbscan/dataset_flickr_hdbscan_puntos.csv",
    index=False, encoding="utf-8"
)

if "Tipo_behavior" in df_export.columns:
    comp = (
        df_export[df_export["cluster_hdbscan"] != -1]
          .groupby(["isla_fix", "cluster_hdbscan"])["Tipo_behavior"]
          .value_counts(normalize=True)
          .unstack()
          .fillna(0)
    )
    comp.to_csv(
        "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_hdbscan/dataset_flickr_hdbscan_comportamiento.csv",
        encoding="utf-8"
    )
print(" Exportados puntos y comportamiento correctamente.")

 Exportados puntos y comportamiento correctamente.


In [34]:
PUNTOS_CSV = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/datasets_hdbscan/dataset_flickr_hdbscan_puntos.csv"
OUT_GPKG   = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/hdbscan/hdbscan_canarias.gpkg"

CRS_WGS84 = "EPSG:4326"
CRS_UTM28 = "EPSG:32628"

df = pd.read_csv(PUNTOS_CSV)

if "latitude" not in df or "longitude" not in df:
    raise ValueError("El CSV no contiene latitude/longitude.")


def nivel_porcentaje(p):
    if p < 0.25:
        return 1
    elif p < 0.50:
        return 2
    elif p < 0.75:
        return 3
    else:
        return 4


print(" Generando niveles turístico/local por cluster…")

records_niveles = []

for (isla, clus), sub in (
    df[df["cluster_hdbscan"] != -1]
    .groupby(["isla_fix", "cluster_hdbscan"])
):


    counts = sub["Tipo_behavior"].value_counts()

    n_fotos_turistico = int(counts.get("Turista", 0))
    n_fotos_local     = int(counts.get("Local", 0))

    total = n_fotos_turistico + n_fotos_local

    p = sub["Tipo_behavior"].value_counts(normalize=True)

    p_turista = p.get("Turista", 0.0)
    p_local   = p.get("Local", 0.0)

    records_niveles.append({
        "isla_fix": isla,
        "cluster_hdbscan": clus,
        "p_turista": p_turista,
        "p_local": p_local,

        "n_fotos_turistico": n_fotos_turistico,
        "n_fotos_local": n_fotos_local,

        "nivel_turistico": nivel_porcentaje(p_turista),
        "nivel_local": nivel_porcentaje(p_local)
    })

tabla_niveles = pd.DataFrame(records_niveles)

df = df.merge(
    tabla_niveles,
    on=["isla_fix", "cluster_hdbscan"],
    how="left"
)

gdf_all = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs=CRS_WGS84
)

gdf_points = gdf_all[gdf_all["cluster_hdbscan"] != -1].copy()
gdf_noise  = gdf_all[gdf_all["cluster_hdbscan"] == -1].copy()


def r90_m(group):
    lat0 = np.radians(group["latitude"].median())
    lon0 = np.radians(group["longitude"].median())
    pts  = np.radians(group[["latitude","longitude"]].values)

    d = np.arccos(
        np.clip(
            np.sin(lat0)*np.sin(pts[:,0]) +
            np.cos(lat0)*np.cos(pts[:,0])*np.cos(pts[:,1] - lon0),
            -1.0, 1.0
        )
    )
    return np.percentile(d * 6371000, 90)

centroid_records = []

for (isla, clus), sub in gdf_points.groupby(["isla_fix","cluster_hdbscan"]):
    centroid_records.append({
    "isla_fix": isla,
    "cluster_hdbscan": clus,

    "nivel_turistico": sub["nivel_turistico"].iloc[0],
    "nivel_local": sub["nivel_local"].iloc[0],

    "n_fotos_turistico": sub["n_fotos_turistico"].iloc[0],
    "n_fotos_local": sub["n_fotos_local"].iloc[0],

    "lat": sub["latitude"].median(),
    "lon": sub["longitude"].median(),
    "n_fotos": len(sub),
    "r90_m": r90_m(sub)
})

gdf_centroids = gpd.GeoDataFrame(
    centroid_records,
    geometry=gpd.points_from_xy(
        [c["lon"] for c in centroid_records],
        [c["lat"] for c in centroid_records]
    ),
    crs=CRS_WGS84
)

cent_utm = gdf_centroids.to_crs(CRS_UTM28)
pts_utm  = gdf_points.to_crs(CRS_UTM28)

buffers = []
for _, r in cent_utm.iterrows():
    buffers.append({
    "isla_fix": r["isla_fix"],
    "cluster_hdbscan": r["cluster_hdbscan"],
    "nivel_turistico": r["nivel_turistico"],
    "nivel_local": r["nivel_local"],

    "n_fotos_turistico": r["n_fotos_turistico"],
    "n_fotos_local": r["n_fotos_local"],

    "n_fotos": r["n_fotos"],
    "r90_m": r["r90_m"],
    "geometry": r.geometry.buffer(float(r["r90_m"]))
})

gdf_buffers = gpd.GeoDataFrame(buffers, crs=CRS_UTM28).to_crs(CRS_WGS84)

hulls = []
for (isla, clus), sub in pts_utm.groupby(["isla_fix","cluster_hdbscan"]):
    geom_union = unary_union(sub.geometry)
    hull = geom_union.convex_hull

    rcent = cent_utm[
        (cent_utm["isla_fix"] == isla) &
        (cent_utm["cluster_hdbscan"] == clus)
    ].iloc[0]

    hulls.append({
    "isla_fix": isla,
    "cluster_hdbscan": clus,
    "nivel_turistico": rcent["nivel_turistico"],
    "nivel_local": rcent["nivel_local"],

    "n_fotos_turistico": rcent["n_fotos_turistico"],
    "n_fotos_local": rcent["n_fotos_local"],

    "n_fotos": rcent["n_fotos"],
    "r90_m": rcent["r90_m"],
    "geometry": hull
})

gdf_hulls = gpd.GeoDataFrame(hulls, crs=CRS_UTM28).to_crs(CRS_WGS84)


spiders = []
cent_idx = {(r["isla_fix"], r["cluster_hdbscan"]): r.geometry for _, r in cent_utm.iterrows()}

for (isla, clus), sub in pts_utm.groupby(["isla_fix","cluster_hdbscan"]):
    cgeom = cent_idx[(isla, clus)]
    for _, pr in sub.iterrows():
        spiders.append({
            "isla_fix": isla,
            "cluster_hdbscan": clus,
            "nivel_turistico": pr["nivel_turistico"],
            "nivel_local": pr["nivel_local"],
            "photo_id": pr.get("id", None),
            "geometry": LineString([cgeom, pr.geometry])
        })

gdf_spider = gpd.GeoDataFrame(spiders, crs=CRS_UTM28).to_crs(CRS_WGS84)


Path(OUT_GPKG).parent.mkdir(parents=True, exist_ok=True)

gdf_centroids.to_file(OUT_GPKG, layer="hdbscan_centroids", driver="GPKG")
gdf_buffers.to_file(  OUT_GPKG, layer="hdbscan_buffers_r90", driver="GPKG")
gdf_hulls.to_file(    OUT_GPKG, layer="hdbscan_hulls",       driver="GPKG")
gdf_points.to_file(   OUT_GPKG, layer="hdbscan_points",      driver="GPKG")
gdf_noise.to_file(    OUT_GPKG, layer="hdbscan_noise",       driver="GPKG")
gdf_spider.to_file(   OUT_GPKG, layer="hdbscan_spider",      driver="GPKG")

print("✅ Exportado GeoPackage HDBSCAN con niveles turístico/local:", OUT_GPKG)

 Generando niveles turístico/local por cluster…
✅ Exportado GeoPackage HDBSCAN con niveles turístico/local: /Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/hdbscan/hdbscan_canarias.gpkg


**GENERAR DATASET PARA EL LLM**

In [35]:
import geopandas as gpd
import pandas as pd

MUNI_PATH = r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\QGIS\municipios\municipios_canarias.gpkg"

municipios = gpd.read_file(MUNI_PATH)

print("Municipios cargados:", len(municipios))
municipios.head()

Municipios cargados: 87


,GEOCODE,GEOPADRE,ETIQUETA,NOTAS,GRANULARID,GCD_PROVIN,GCD_ISLA,GCD_GRANCO,GCD_COMARC,IGN_SUP,IGN_PERIM,UTM_X,UTM_Y,LONGITUD,LATITUD,UTM_X_CAPI,UTM_Y_CAPI,LONG_CAPI,LATI_CAPI,geometry
0,35001,ES705A22,Agaete,Ayuntamiento de Agaete,MUNICIPIOS,ES701,ES705,ES705A2,ES705A22,4451.16,41038.0,432237.32,3105495.86,-15.689645,28.073135,431360.28,3108387.71,-15.698739,28.099193,"MULTIPOLYGON (((-15.74604 28.05081, -15.74522 ..."
1,35002,ES705A32,Agüimes,Ayuntamiento de Agüimes,MUNICIPIOS,ES701,ES705,ES705A3,ES705A32,7889.72,55508.0,455417.29,3085973.99,-15.453004,27.897893,456117.58,3086639.88,-15.445914,27.903927,"MULTIPOLYGON (((-15.40839 27.84837, -15.40836 ..."
2,35003,ES704A01,Antigua,Ayuntamiento De Antigua,MUNICIPIOS,ES701,ES704,ES704A0,ES704A01,25024.80,79292.0,603336.94,3136732.92,-13.945579,28.352768,596564.74,3144502.79,-14.014024,28.423412,"MULTIPOLYGON (((-13.91938 28.25134, -13.91945 ..."
3,35004,ES708A01,Arrecife,Ayuntamiento De Arrecife,MUNICIPIOS,ES701,ES708,ES708A0,ES708A01,2427.54,36641.0,641157.78,3206510.34,-13.551117,28.978878,641613.10,3204338.91,-13.546719,28.959235,"MULTIPOLYGON (((-13.58361 28.95211, -13.58426 ..."
4,35005,ES705A23,Artenara,Ayuntamiento de Artenara,MUNICIPIOS,ES701,ES705,ES705A2,ES705A23,6641.90,50791.0,432859.41,3099392.63,-15.682966,28.018075,436451.97,3099620.39,-15.646436,28.020308,"MULTIPOLYGON (((-15.74959 27.98801, -15.75005 ..."


In [36]:
import geopandas as gpd

GPKG_PATH = r"/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/QGIS/hdbscan/hdbscan_canarias.gpkg"

gdf_hulls = gpd.read_file(GPKG_PATH, layer="hdbscan_hulls")

In [37]:
municipios = municipios.to_crs(gdf_hulls.crs)

In [38]:
join = gpd.sjoin(
    gdf_hulls,
    municipios,
    how="inner",
    predicate="intersects"
)

print("Registros intersectados:", len(join))

Registros intersectados: 1045


In [39]:
COLUMNA_MUNICIPIO = "ETIQUETA"

def nivel_municipal(nivel_medio):
    if nivel_medio < 1.75:
        return 1
    elif nivel_medio < 2.50:
        return 2
    elif nivel_medio < 3.25:
        return 3
    else:
        return 4

rows = []

for municipio, sub in join.groupby(COLUMNA_MUNICIPIO):

    municipio_clean = municipio.strip()
    if municipio_clean.lower() == "frontera (hasta 2007)":
        municipio_clean = "Frontera"
    elif municipio_clean.lower() == "el tanque":
        municipio_clean = "Tanque"
    elif municipio_clean.lower() == "vilaflor de chasna":
        municipio_clean = "Vilaflor"

    niveles = (
        sub.groupby("cluster_hdbscan")["nivel_turistico"]
        .first()
        .values
    )

    if len(niveles) == 0:
        continue

    nivel_turistico_medio = float(np.mean(niveles))
    nivel_turistico_municipio = nivel_municipal(nivel_turistico_medio)

    cluster_ids = sorted(sub["cluster_hdbscan"].unique())

    tmp = sub.drop_duplicates("cluster_hdbscan")

    n_fotos_turistico = int(tmp["n_fotos_turistico"].fillna(0).sum())
    n_fotos_local     = int(tmp["n_fotos_local"].fillna(0).sum())


    rows.append({
        "municipio": municipio_clean,
        "isla": sub["isla_fix"].iloc[0],

        "nivel_turistico_municipio": nivel_turistico_municipio,
        "nivel_turistico_medio": round(nivel_turistico_medio, 2),

        "n_clusters": len(niveles),
        "clusters_ids": ", ".join(map(str, cluster_ids)),

        "n_fotos_turistico": n_fotos_turistico,
        "n_fotos_local": n_fotos_local
    })

df_municipios_cluster = pd.DataFrame(rows)

OUT_CSV = "/Users/cgsos/Documents/Cuarto/TFG/FLICKR/datos/LLM/municipios_etiquetados_niveles.csv"
df_municipios_cluster.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("✅ Dataset de municipios generado correctamente con nº de fotos locales y turísticas")

✅ Dataset de municipios generado correctamente con nº de fotos locales y turísticas


In [13]:
df_municipios_cluster["municipio"].unique()

<StringArray>
[                              'Adeje',                              'Agaete',
                               'Agulo',                             'Agüimes',
                             'Alajeró',                             'Antigua',
                               'Arafo',                               'Arico',
                               'Arona',                            'Arrecife',
                            'Artenara',                              'Arucas',
                          'Barlovento',                          'Betancuria',
                          'Breña Alta',                          'Breña Baja',
                'Buenavista del Norte',                          'Candelaria',
                             'El Paso',                          'El Rosario',
                           'El Sauzal',                              'Tanque',
                              'Fasnia',                              'Firgas',
                            'Frontera'

In [15]:
df_municipios_cluster.groupby(["isla", "nivel_turistico_municipio"]).size()

isla           nivel_turistico_municipio
El Hierro      4                             2
Fuerteventura  1                             1
               2                             1
               3                             3
               4                             1
Gran Canaria   1                             1
               2                             4
               3                            12
               4                             2
La Gomera      2                             1
               3                             4
               4                             1
La Palma       3                             3
               4                            11
Lanzarote      1                             1
               2                             4
               3                             1
               4                             1
Tenerife       1                             1
               2                             7
               3   